# 01 — Data Pipeline: Ingestion, Storage & Quality

**Series:** Piccolo ML Options Strategy Research  
**Depends on:** `00_experiment_plan.ipynb`


## 1. Overview

This notebook documents and validates the full data pipeline that feeds the
Piccolo strategy.  We cover:

1. How IBKR EOD prices are ingested (bootstrap + daily top-up)
2. How IBKR options chain snapshots are captured
3. How historical CBOE data is incorporated
4. DuckDB schema — tables, row counts, date ranges
5. Data quality checks — nulls, gaps, outliers
6. Sample visualisations — price history, open interest over time


## 2. Environment Setup


In [ ]:
%matplotlib inline
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import duckdb
import warnings
warnings.filterwarnings("ignore")

# ── Path setup ──────────────────────────────────────────────────────────────
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from config.settings import DUCKDB_PATH_HIST, DUCKDB_PATH_LIVE

print(f"DUCKDB_PATH_HIST : {DUCKDB_PATH_HIST}")
print(f"DUCKDB_PATH_LIVE : {DUCKDB_PATH_LIVE}")


## 3. Ingestion Architecture

### 3.1 IBKR EOD Prices

`bootstrap_eod_prices_ibkr.py` performs a one-time historical backfill via
the IBKR API (`reqHistoricalData`).  It stores OHLCV bars for each symbol in
the `DUCKDB_PATH_LIVE` database.

`eod_prices_daily_ibkr.py` runs nightly (or via cron) to append the most
recent session's bars.  Both scripts write to the same DuckDB table, using
`INSERT OR REPLACE` semantics to prevent duplicates.

```
IBKR TWS/Gateway
      │
      ▼  reqHistoricalData (1d bars)
bootstrap_eod_prices_ibkr.py  ──►  DuckDB LIVE → table: eod_prices
eod_prices_daily_ibkr.py      ──►  DuckDB LIVE → table: eod_prices  (daily top-up)
```

### 3.2 IBKR Options Chain Snapshots

`ibkr_options_snapshot.py` captures a full options chain snapshot (all strikes
× expiries) for each live symbol.  It computes and stores:
- OI per strike/expiry
- Implied volatility
- GEX contribution per strike (dealer gamma × OI × contract multiplier)

Snapshots are written to `DUCKDB_PATH_LIVE_OPTIONS`.

### 3.3 Historical CBOE Data

14 years of CBOE data provides the deep historical backbone for model training.
It is pre-processed and merged into the historical feature table stored in
`DUCKDB_PATH_HIST`.


## 4. DuckDB Schema Exploration


In [ ]:
# ── Connect to historical DB ─────────────────────────────────────────────────
con_hist = duckdb.connect(DUCKDB_PATH_HIST, read_only=True)

print("=== Historical DB — Tables ===")
tables_hist = con_hist.execute("SHOW TABLES").fetchdf()
print(tables_hist.to_string(index=False))


In [ ]:
# ── Row counts and date ranges for each table ────────────────────────────────
for tbl in tables_hist["name"]:
    try:
        sql = (
            f"SELECT COUNT(*) AS row_count, "
            f"MIN(date) AS min_date, MAX(date) AS max_date, "
            f"COUNT(DISTINCT symbol) AS n_symbols "
            f"FROM {tbl}"
        )
        info = con_hist.execute(sql).fetchdf()
        print(f"\n{tbl}:")
        print(info.to_string(index=False))
    except Exception as e:
        print(f"\n{tbl}: could not query — {e}")


In [ ]:
# ── Connect to live DB ───────────────────────────────────────────────────────
try:
    con_live = duckdb.connect(DUCKDB_PATH_LIVE, read_only=True)
    print("=== Live DB — Tables ===")
    tables_live = con_live.execute("SHOW TABLES").fetchdf()
    print(tables_live.to_string(index=False))
except Exception as e:
    print(f"Live DB not available: {e}")


### 4.1 Expected Schema

```
eod_prices
  date        DATE        Trading date
  symbol      VARCHAR     Ticker (e.g. 'SPY')
  open        DOUBLE
  high        DOUBLE
  low         DOUBLE
  close       DOUBLE
  volume      BIGINT

option_chains
  trade_date  DATE        Snapshot date
  symbol      VARCHAR
  expiry      VARCHAR     Expiry in YYYYMMDD format
  strike      DOUBLE
  righttype   VARCHAR     'C' or 'P'
  open_interest DOUBLE
  iv          DOUBLE
  delta       DOUBLE
  gamma       DOUBLE

features_spy_latest
  quote_date  DATE
  < all feature columns — see Notebook 02 >
  und_price   DOUBLE      Underlying SPY price
```


## 5. Data Quality Checks


In [ ]:
# ── 5.1 Load SPY EOD prices ──────────────────────────────────────────────────
try:
    spy_prices = con_hist.execute(
        "SELECT date, open, high, low, close, volume "
        "FROM eod_prices WHERE symbol = 'SPY' ORDER BY date"
    ).fetchdf()
    spy_prices["date"] = pd.to_datetime(spy_prices["date"])
    print(f"SPY price rows : {len(spy_prices):,}")
    print(f"Date range     : {spy_prices['date'].min().date()} to {spy_prices['date'].max().date()}")
    print(f"Null counts:\n{spy_prices.isnull().sum()}")
except Exception as e:
    print(f"Could not load SPY prices: {e}")
    # TODO: paste your actual output here


In [ ]:
# ── 5.2 Trading day gaps check ───────────────────────────────────────────────
try:
    spy_prices = spy_prices.sort_values("date").reset_index(drop=True)
    spy_prices["prev_date"] = spy_prices["date"].shift(1)
    spy_prices["gap_days"] = (spy_prices["date"] - spy_prices["prev_date"]).dt.days

    # Calendar gaps > 5 days (excluding weekends + typical holidays) are suspicious
    large_gaps = spy_prices[spy_prices["gap_days"] > 5].dropna()
    print(f"Large date gaps (>5 calendar days): {len(large_gaps)}")
    if len(large_gaps) > 0:
        print(large_gaps[["date", "prev_date", "gap_days"]].to_string(index=False))
    else:
        print("No suspicious gaps found.")
except NameError:
    print("spy_prices not loaded — run the cell above first.")
    # TODO: paste your actual output here


In [ ]:
# ── 5.3 Outlier detection on daily returns ───────────────────────────────────
try:
    spy_prices["ret_1d"] = spy_prices["close"].pct_change()
    outliers = spy_prices[spy_prices["ret_1d"].abs() > 0.05]
    print(f"Days with |1d return| > 5%: {len(outliers)}")
    print(outliers[["date", "close", "ret_1d"]].to_string(index=False))
except NameError:
    print("spy_prices not loaded — run earlier cells first.")
    # TODO: paste your actual output here


## 6. Sample Visualisations

### 6.1 SPY Price History


In [ ]:
try:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1]})

    ax1, ax2 = axes

    # Price
    ax1.plot(spy_prices["date"], spy_prices["close"], color="#20808D", linewidth=1.2,
             label="SPY Close")
    ax1.set_ylabel("Price (USD)", fontsize=11)
    ax1.set_title("SPY — Historical EOD Close Price", fontsize=13, fontweight="bold")
    ax1.legend(loc="upper left")
    ax1.grid(alpha=0.3)

    # Volume
    ax2.bar(spy_prices["date"], spy_prices["volume"], color="#20808D", alpha=0.5,
            width=1, label="Volume")
    ax2.set_ylabel("Volume", fontsize=11)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax2.xaxis.set_major_locator(mdates.YearLocator())
    ax2.grid(alpha=0.3)

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.savefig("spy_price_history.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Chart saved to spy_price_history.png")
except NameError:
    print("spy_prices not loaded — run earlier cells first.")
    # TODO: paste your actual output here


### 6.2 Open Interest Over Time (Options Snapshot)


In [ ]:
# ── Aggregate daily total OI from options snapshots ──────────────────────────
try:
    oi_sql = (
        "SELECT CAST(trade_date AS DATE) AS snap_date, "
        "SUM(OpenInterest) AS total_oi "
        "FROM option_chains WHERE Symbol = 'SPY' "
        "GROUP BY 1 ORDER BY 1"
    )
    con_opt = duckdb.connect(os.getenv('DUCKDB_PATH_LIVE_OPTIONS'), read_only=True)
    oi_daily = con_opt.execute(oi_sql).fetchdf()
    oi_daily["snap_date"] = pd.to_datetime(oi_daily["snap_date"])

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(oi_daily["snap_date"], oi_daily["total_oi"] / 1e6,
            color="#A84B2F", linewidth=1.2)
    ax.set_ylabel("Total Open Interest (M contracts)", fontsize=11)
    ax.set_title("SPY — Aggregate Options Open Interest Over Time", fontsize=13, fontweight="bold")
    ax.grid(alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.savefig("spy_oi_history.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"OI chart skipped: {e}")
    # TODO: paste your actual output here


## 7. Findings & Notes

> **TODO:** Fill in after running this notebook.

| Check | Result | Action |
|-------|--------|--------|
| EOD price row count | _paste here_ | |
| Date range | _paste here_ | |
| Null values | _paste here_ | |
| Large date gaps | _paste here_ | |
| Extreme return outliers | _paste here_ | |
| OI availability | _paste here_ | |

## 8. Next

Proceed to [02_feature_engineering.ipynb](02_feature_engineering.ipynb) to
inspect the constructed feature table and label distribution.
